# Module `metric_closure.py`

Ce notebook explique la **fermeture metrique** d'un graphe et detaille pourquoi cette brique est volontairement separee du generateur.

**Definition.** La fermeture metrique d'un graphe pondere `G = (V, E, w)` est le **graphe complet** `G* = (V, E*, d)` ou `d(i, j)` est la longueur du **plus court chemin** entre `i` et `j` dans `G`. Quand `G` est connexe avec poids positifs, `G*` est par construction **metrique** : il satisfait l'inegalite triangulaire (Ahuja, Magnanti & Orlin [1, chap. 4]).

**Pourquoi en a-t-on besoin ?** Un solveur VRP/TSP veut pouvoir evaluer le cout d'une tournee `[v0, v1, v2, ...]` en sommant `cout(v0,v1) + cout(v1,v2) + ...` - peu importe que ces aretes existent vraiment dans le graphe routier. Or notre graphe residuel a beaucoup de paires de sommets sans arete directe (densite faible, contraintes FORBIDDEN). Sans la fermeture metrique, le solveur devrait re-executer un Dijkstra a chaque evaluation de cout, soit **des millions de fois** dans une boucle de metaheuristique - inutilisable.

**Pourquoi un module separe** plutot qu'integre au generateur ? Cette brique est reutilisee par :

- les **solveurs** qui ont besoin d'une matrice complete entre sommets (consommee via `SolverInput`) ;
- la **simulation dynamique** qui doit recalculer la matrice a chaque tour (les couts changent, donc les plus courts chemins aussi) ;
- l'**execution dynamique** `execute_dynamic` qui trace le chemin reel pris par un vehicule pour detecter les conflits avec les nouvelles aretes FORBIDDEN.

Une seule implementation, trois consommateurs : c'est exactement la situation qui justifie un module a part. Le module contient des primitives reutilisables (`dijkstra`, `reconstruct_path`, `normalize_edge`, etc.) et une fonction haut-niveau `complete_graph_with_shortest_paths` qui les compose.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.metric_closure import (
    build_cost_matrix,
    build_neighbor_map,
    check_triangle_inequality,
    complete_graph_with_shortest_paths,
    dijkstra,
    normalize_edge,
    reconstruct_path,
)

## 1. Role du module

Le graphe residuel peut avoir des paires de sommets sans arete directe (densite faible, arcs FORBIDDEN). Pour un solveur TSP/VRP metrique, on veut pourtant un cout entre chaque paire.

`metric_closure.py` produit :

- **`completed_costs`** : matrice `n x n` des plus courts chemins. C'est la matrice de distances **metrique** que tous les algorithmes consomment.
- **`completed_paths`** : dictionnaire `{(i, j): [i, ..., j]}` qui donne la vraie sequence de sommets traversee. On ne peut pas la reconstruire a partir de la seule matrice, donc on la retourne explicitement.

**Garantie metrique**. La matrice respecte l'inegalite triangulaire par construction, consequence directe de Dijkstra : si un chemin de longueur `d(i,k) + d(k,j) < d(i,j)` existait, ce serait **lui** le plus court chemin de `i` a `j`, contradiction. Cette propriete permet d'appliquer sans restriction les heuristiques qui s'appuient dessus (2-opt, Christofides [2], Split de Prins [3]).

## 2. `normalize_edge` : cle canonique

Le graphe est **non oriente** : l'arete entre `u` et `v` est la meme que celle entre `v` et `u`. Mais Python ne le sait pas : `(0, 3)` et `(3, 0)` sont deux tuples differents, donc deux cles differentes dans un dictionnaire.

**Solution standard** : adopter une **forme canonique** unique. `normalize_edge` impose `(min(u, v), max(u, v))`, donc `normalize_edge(0, 3) == normalize_edge(3, 0) == (0, 3)`. C'est une convention mineure mais qui evite des bugs subtils :

- doublons silencieux dans les iterations sur `edge_costs.items()` ;
- mises a jour qui ne s'appliquent pas (ecriture en `(u, v)`, lecture en `(v, u)`) ;
- comparaisons de presence/absence qui retournent des resultats incoherents.

**Pourquoi pas un `frozenset({u, v})` plutot qu'un tuple ordonne ?** Parce qu'un tuple est **ordonnable** (donc utilisable dans `sorted(edges)`) et plus rapide a hasher qu'un frozenset. La convention `(min, max)` donne en plus une lecture intuitive ("aretes triees par origine puis destination").

In [2]:
print(normalize_edge(3, 0))
print(normalize_edge(0, 3))
print(normalize_edge(4, 4))

(0, 3)
(0, 3)
(4, 4)


## 3. `build_cost_matrix` : dictionnaire -> matrice dense

Prend un `dict[(u,v), cout]` et produit la matrice symetrique `n x n`. Les paires absentes ou a cout `inf` laissent la case a `0.0`.

**Pourquoi cette conversion ?** Parce qu'on a deux representations qui ont chacune leurs forces :

- **Dictionnaire d'aretes** : naturel pour modifier le graphe (ajouter/supprimer des aretes), efficace en memoire pour les graphes peu denses (ne stocke que les aretes existantes).
- **Matrice dense** : acces O(1) a `matrix[i][j]`, indispensable dans les boucles internes des solveurs (calcul de `delta` differentiel des operateurs de voisinage).

**Pourquoi `0.0` pour les paires absentes plutot que `inf` ou `None` ?** Parce que la matrice produite ici represente le graphe **brut** (base ou residuel), pas la fermeture metrique. Un solveur ne devrait jamais lire ces cellules sans passer par la fermeture metrique d'abord. La valeur `0.0` est un placeholder sentinelle qui rend evident un bug si elle est lue par erreur (un cout 0 entre deux sommets distincts est manifestement faux). La fermeture metrique remplace ces 0 par les vraies distances.

In [3]:
edge_costs = {(0, 1): 4.0, (1, 2): 3.0, (0, 2): 10.0}
for row in build_cost_matrix(3, edge_costs):
    print(row)

[0.0, 4.0, 10.0]
[4.0, 0.0, 3.0]
[10.0, 3.0, 0.0]


## 4. `build_neighbor_map` : format adapte a Dijkstra

Transforme le dict d'aretes en **listes d'adjacence** : `{sommet: [(voisin, cout), ...]}`.

**Pourquoi ce format pour Dijkstra ?** Parce que Dijkstra a besoin, pour chaque sommet extrait du tas, d'**iterer rapidement sur ses voisins**. Avec un dictionnaire global d'aretes, retrouver les voisins d'un sommet `u` necessite de scanner toutes les aretes O(m) - faisable mais inutilement couteux. Avec des listes d'adjacence, c'est O(deg(u)). Sur un graphe a `n` sommets et `m` aretes, le cout total de Dijkstra passe de O(n*m + n*log n) a O((n+m)*log n) - c'est l'analyse classique de Cormen et al. [4, chap. 24].

**Filtrage des aretes FORBIDDEN.** Les aretes a cout `inf` (interdites) sont supprimees a la **construction** du neighbor map plutot que filtrees pendant Dijkstra. Resultat : Dijkstra ne les voit jamais et n'a aucun test conditionnel a faire dans sa boucle interne. C'est une optimisation classique : pousser les filtres en amont du hot loop. La verification "is_forbidden" devient ainsi gratuite par construction.

In [4]:
neighbors = build_neighbor_map(3, edge_costs)
for node, nbrs in neighbors.items():
    print(node, "->", nbrs)

0 -> [(1, 4.0), (2, 10.0)]
1 -> [(0, 4.0), (2, 3.0)]
2 -> [(1, 3.0), (0, 10.0)]


## 5. `dijkstra` : plus courts chemins depuis une source

**Algorithme historique** : Dijkstra [5] (1959). Calcule les plus courts chemins depuis **une source** vers **tous** les autres sommets, en supposant des **poids positifs** (pour les poids negatifs, il faudrait Bellman-Ford). Ici on a des distances euclidiennes plus des surcouts multiplicatifs - tous strictement positifs - donc Dijkstra est applicable.

**Implementation classique a tas binaire** (`heapq` Python) :

- file de priorite sur `(cout_cumule, sommet)` : on extrait toujours le sommet **non encore traite** de plus petit cout connu ;
- on ignore un pop si `current_cost > distances[node]` : ca veut dire qu'on a deja traite ce sommet via un chemin plus court, l'entree dans le tas est obsolete (technique de **lazy deletion**, voir Cormen et al. [4, section 24.3] - evite la complication d'une primitive `decrease_key`) ;
- on met a jour `predecessors[neighbor]` quand on **relache** une arete (i.e. quand on ameliore la distance vers un voisin), pour pouvoir reconstruire les chemins plus tard.

**Complexite : O((n + m) * log n)** avec un tas binaire. Suffisant pour les tailles cibles du projet (jusqu'a quelques centaines de sommets). Pour aller plus loin, il faudrait un tas de Fibonacci (Fredman & Tarjan [6], O((n log n) + m), mais constante elevee qui le rend rarement plus rapide en pratique avant n >> 10000).

In [5]:
distances, predecessors = dijkstra(0, 3, neighbors)
print("Distances depuis 0 :", distances)
print("Predecesseurs     :", predecessors)

Distances depuis 0 : [0.0, 4.0, 7.0]
Predecesseurs     : [None, 0, 1]


## 6. `reconstruct_path` : remonter la trace

A partir du tableau `predecessors`, on reconstruit le chemin `source -> target` en remontant de `target` vers `source` puis en inversant.

**Pourquoi pas stocker directement les chemins pendant Dijkstra ?** Parce que ce serait O(n^2) en memoire dans le pire cas (chaque sommet stockerait son chemin complet). Le tableau `predecessors` est O(n) et permet de reconstruire **n'importe quel** chemin a la demande en O(longueur du chemin). C'est l'astuce standard pour separer le **calcul** (Dijkstra renvoie une trace minimale) de la **lecture** (a la demande, en parcourant la trace).

**Cas particuliers** :

- `source == target` : renvoie `[source]` (chemin trivial de longueur 0).
- `target` inatteignable : renvoie `[]` (distance infinie, predecessor = None). Pour notre generateur c'est impossible par construction (graphe garantit connexe, voir `graph_generator.ipynb` section 5), mais le module reste robuste a un usage hors generateur.

Ce tableau de predecesseurs est produit a chaque Dijkstra ; `reconstruct_path` est juste un accesseur.

In [6]:
print("Chemin 0 -> 2 :", reconstruct_path(0, 2, predecessors))
print("Chemin 0 -> 0 :", reconstruct_path(0, 0, predecessors))

Chemin 0 -> 2 : [0, 1, 2]
Chemin 0 -> 0 : [0]


## 7. `complete_graph_with_shortest_paths` : haut niveau

Cette fonction est la facade du module : c'est elle que le generateur, le simulateur dynamique et les tests appellent dans le flux normal. Elle compose les briques vues avant dans un pipeline volontairement simple :

1. transformer le dictionnaire d'aretes actives en listes d'adjacence avec `build_neighbor_map` ;
2. lancer `dijkstra(source, ...)` pour chaque sommet source ;
3. copier les distances obtenues dans une matrice dense `matrix[source][target]` ;
4. reconstruire les chemins reels avec `reconstruct_path` et les stocker dans `paths`.

**Strategie algorithmique.** On resout un APSP (*All-Pairs Shortest Paths*) en repetant un SSSP (*Single-Source Shortest Paths*). Avec un tas binaire, la complexite totale vaut `O(n * (n + m) * log n)`. Sur les graphes CESIPATH, `m` reste volontairement limite par les profils de densite du generateur : on est donc plus proche d'un graphe creux que d'un graphe complet physique. C'est precisement le cas ou repeter Dijkstra est rentable.

**Pourquoi pas Floyd-Warshall** [7] ? Floyd-Warshall est elegant et tres court a coder, mais il paie toujours `O(n^3)`, meme si le graphe contient peu d'aretes. Ici on veut recalculer la fermeture metrique potentiellement a chaque tour dynamique ; le cout doit donc suivre le nombre d'aretes actives. De plus, Dijkstra produit naturellement un tableau de predecesseurs par source, ce qui rend `completed_paths` presque gratuit a obtenir. Avec Floyd-Warshall, il faudrait maintenir une matrice `next` ou `predecessor` en plus et faire attention aux mises a jour intermediaires.

**Pourquoi stocker uniquement `paths[(s, t)]` pour `s < t` ?** Les chemins sont symetriques dans un graphe non oriente : le chemin `3 -> 0` est le chemin `0 -> 3` lu a l'envers. Stocker les deux sens doublerait la memoire sans ajouter d'information. Le reste du projet utilise cette convention avec des cles normalisees.

**Attention au sens metier.** La matrice `completed_costs` ne dit pas qu'une arete directe existe entre deux clients. Elle dit : "si le solveur decide de passer de `i` a `j`, voici le meilleur trajet routier reel disponible maintenant". La difference est essentielle en dynamique : une solution VRP travaille sur la matrice complete, mais l'execution doit pouvoir redescendre au chemin reel pour verifier quelles vraies routes sont traversees.


In [7]:
node_count = 4
edge_costs = {(0, 1): 4.0, (1, 2): 3.0, (2, 3): 5.0, (0, 2): 10.0}

completed_costs, completed_paths = complete_graph_with_shortest_paths(node_count, edge_costs)
print("Matrice complete :")
for row in completed_costs:
    print(row)

print("\nChemins reels :")
for key, path in sorted(completed_paths.items()):
    print(f"  {key} -> {path}")

Matrice complete :
[0.0, 4.0, 7.0, 12.0]
[4.0, 0.0, 3.0, 8.0]
[7.0, 3.0, 0.0, 5.0]
[12.0, 8.0, 5.0, 0.0]

Chemins reels :
  (0, 1) -> [0, 1]
  (0, 2) -> [0, 1, 2]
  (0, 3) -> [0, 1, 2, 3]
  (1, 2) -> [1, 2]
  (1, 3) -> [1, 2, 3]
  (2, 3) -> [2, 3]


## 8. `check_triangle_inequality` : garde-fou

La fermeture metrique devrait toujours respecter :

```text
d(i, j) <= d(i, k) + d(k, j)
```

pour tous les triplets `(i, j, k)`. Intuition : si passer par `k` etait moins cher que le cout annonce entre `i` et `j`, alors le cout annonce n'etait pas un plus court chemin.

La fonction est donc moins un outil de production qu'un **test d'invariant**. Elle sert a detecter trois familles de problemes :

- une matrice construite a la main et pas vraiment metrique ;
- une erreur de symetrie ou d'arrondi dans une transformation ;
- une regression dans `complete_graph_with_shortest_paths` ou dans le filtrage des aretes actives.

Le triple parcours coute `O(n^3)`. Ce n'est pas un souci pour un garde-fou appele ponctuellement dans un notebook ou un test, mais ce n'est pas une fonction a placer dans la boucle interne d'une metaheuristique.

La tolerance `1e-9` absorbe les micro-erreurs flottantes. Les couts sont arrondis a deux decimales dans le projet, mais les additions de floats peuvent produire des valeurs comme `12.300000000000002`; la tolerance evite de signaler une fausse violation.


In [8]:
ok, violation = check_triangle_inequality(completed_costs)
print("Inegalite respectee ?", ok)
print("Violation         :", violation)

Inegalite respectee ? True
Violation         : None


## 10. Pourquoi un module dedie ?

Isoler Dijkstra et la fermeture metrique rend l'architecture plus lisible et plus evolutive.

**Dans le generateur**, le module transforme un graphe residuel partiel en instance exploitable par les solveurs. **Dans la dynamique**, il reconstruit la matrice complete apres chaque changement de cout ou disponibilite. **Dans l'execution**, il garde le lien entre decision abstraite (`aller de 2 a 7`) et trajet routier reel (`2 -> 4 -> 1 -> 7`).

Cette separation protege aussi les algorithmes. GRASP, SA, tabou et genetique n'ont pas besoin de connaitre les aretes physiques, les statuts `FORBIDDEN`, ni les detours imposes par le reseau. Ils recoivent une matrice metrique propre via `SolverInput`. C'est ce contrat qui permet de comparer les metaheuristiques sans melanger optimisation VRP et logique reseau.

Evolutions possibles sans casser le reste :

- remplacer Dijkstra par A* si on exploite les coordonnees comme heuristique admissible ;
- brancher un backend optimise si les instances deviennent beaucoup plus grandes ;
- ajouter un cache incremental si la dynamique modifie peu d'aretes entre deux tours ;
- exposer des diagnostics plus fins sur les paires dont le chemin reel change souvent.

**A retenir** : `metric_closure.py` est la couche de traduction entre un reseau routier incomplet et les matrices metriques attendues par les solveurs. Sans elle, les algorithmes seraient forces de refaire du plus court chemin partout ; avec elle, ils travaillent en `O(1)` pour lire un cout entre deux sommets.

---

## References

[1] **Ahuja, R. K., Magnanti, T. L. & Orlin, J. B. (1993).** *Network Flows: Theory, Algorithms, and Applications.* Prentice Hall.

[2] **Christofides, N. (1976).** *Worst-case analysis of a new heuristic for the travelling salesman problem.* Technical Report 388, Carnegie Mellon University.

[3] **Prins, C. (2004).** *A simple and effective evolutionary algorithm for the vehicle routing problem.* Computers & Operations Research, 31(12), 1985-2002.

[4] **Cormen, T. H., Leiserson, C. E., Rivest, R. L. & Stein, C. (2022).** *Introduction to Algorithms* (4th ed.). MIT Press.

[5] **Dijkstra, E. W. (1959).** *A note on two problems in connexion with graphs.* Numerische Mathematik, 1, 269-271.

[6] **Fredman, M. L. & Tarjan, R. E. (1987).** *Fibonacci heaps and their uses in improved network optimization algorithms.* Journal of the ACM, 34(3), 596-615.

[7] **Floyd, R. W. (1962).** *Algorithm 97: Shortest Path.* Communications of the ACM, 5(6), 345.

## 9. Pourquoi un module dedie ?

Isoler Dijkstra et la fermeture metrique rend l'architecture plus lisible et plus evolutive.

**Dans le generateur**, le module transforme un graphe residuel partiel en instance exploitable par les solveurs. **Dans la dynamique**, il reconstruit la matrice complete apres chaque changement de cout ou disponibilite. **Dans l'execution**, il garde le lien entre decision abstraite (`aller de 2 a 7`) et trajet routier reel (`2 -> 4 -> 1 -> 7`).

Cette separation protege aussi les algorithmes. GRASP, SA, tabou et genetique n'ont pas besoin de connaitre les aretes physiques, les statuts `FORBIDDEN`, ni les detours imposes par le reseau. Ils recoivent une matrice metrique propre via `SolverInput`. C'est ce contrat qui permet de comparer les metaheuristiques sans melanger optimisation VRP et logique reseau.

Evolutions possibles sans casser le reste :

- remplacer Dijkstra par A* si on exploite les coordonnees comme heuristique admissible ;
- brancher un backend optimise si les instances deviennent beaucoup plus grandes ;
- ajouter un cache incremental si la dynamique modifie peu d'aretes entre deux tours ;
- exposer des diagnostics plus fins sur les paires dont le chemin reel change souvent.

**A retenir** : `metric_closure.py` est la couche de traduction entre un reseau routier incomplet et les matrices metriques attendues par les solveurs. Sans elle, les algorithmes seraient forces de refaire du plus court chemin partout ; avec elle, ils travaillent en `O(1)` pour lire un cout entre deux sommets.

---

## References

[1] **Ahuja, R. K., Magnanti, T. L. & Orlin, J. B. (1993).** *Network Flows: Theory, Algorithms, and Applications.* Prentice Hall.

[2] **Christofides, N. (1976).** *Worst-case analysis of a new heuristic for the travelling salesman problem.* Technical Report 388, Carnegie Mellon University.

[3] **Prins, C. (2004).** *A simple and effective evolutionary algorithm for the vehicle routing problem.* Computers & Operations Research, 31(12), 1985-2002.

[4] **Cormen, T. H., Leiserson, C. E., Rivest, R. L. & Stein, C. (2022).** *Introduction to Algorithms* (4th ed.). MIT Press.

[5] **Dijkstra, E. W. (1959).** *A note on two problems in connexion with graphs.* Numerische Mathematik, 1, 269-271.

[6] **Fredman, M. L. & Tarjan, R. E. (1987).** *Fibonacci heaps and their uses in improved network optimization algorithms.* Journal of the ACM, 34(3), 596-615.

[7] **Floyd, R. W. (1962).** *Algorithm 97: Shortest Path.* Communications of the ACM, 5(6), 345.
